# 05 — Analysis of the final models

Final configuration: sum of the two embeddings, attention pooling over the genes of each pathway,
mean over pathways, embedding size 64. Three seeds on TCGA and three on GTEx, always evaluated on
the fixed test split.

Heavy results are saved in `results/analysis/`. When they are there, they are read from disk and the
models are not loaded, so the notebook runs in seconds.

In [ ]:
import json, os, itertools, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score, f1_score, log_loss

import kghtt_model_v5 as M
import kghtt_analysis as A

RESULTS = "results"
OUT = os.path.join(RESULTS, "analysis")
os.makedirs(OUT, exist_ok=True)
DATASETS, SEEDS, TAG = ("tcga", "gtex"), (0, 1, 2), "d64"
RUNS = {f"{ds}_s{s}": os.path.join(RESULTS, f"{ds}_sum_gene-attn_s{s}_{TAG}") for ds in DATASETS for s in SEEDS}

N_PERM, BATCH = 10, 64       # only used if a heavy result is not on disk yet
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", message="An input array is constant")

# light information saved at training time: metrics and test predictions
info = {}
for key, path in RUNS.items():
    run = json.load(open(os.path.join(path, "run.json")))
    pred = np.load(os.path.join(path, "predictions.npz"))
    info[key] = dict(dataset=run["args"]["dataset"], seed=run["args"]["seed"], metrics=run["metrics"],
                     classes=run["classes"], y=pred["y_test"], p=pred["p_test"], confusion=pred["confusion"])
    print(f"{key}: {len(info[key]['classes'])} classes, {len(info[key]['y'])} test samples, "
          f"balanced accuracy {run['metrics']['balanced_accuracy']:.4f}")

_models = {}
def get_run(key):
    """Load a trained model only when a heavy result has to be computed."""
    if key not in _models:
        r = A.load_run(RUNS[key], M)
        ok, err, _ = A.check_reload(r)
        print(f"  {key}: model loaded, predictions {'match' if ok else 'DO NOT match'} the saved ones ({err:.1e})")
        _models[key] = r
    return _models[key]

def keys_of(ds):
    return [k for k in RUNS if info[k]["dataset"] == ds]

def baseline(ds, name):
    d = os.path.join(RESULTS, "baselines", ds, name)
    if not os.path.exists(os.path.join(d, "run.json")):
        return None
    return dict(metrics=json.load(open(os.path.join(d, "run.json")))["metrics"],
                confusion=np.load(os.path.join(d, "predictions.npz"))["confusion"])

## 1. Performance

Balanced accuracy and log-loss on the test set. AUC is reported but it is above 0.995 for every
model, so it does not separate them. *Ensemble* = average of the predicted probabilities of the
three seeds.

In [ ]:
rows = []
for ds in DATASETS:
    ks = keys_of(ds)
    y, C = info[ks[0]]["y"], len(info[ks[0]]["classes"])
    m = pd.DataFrame([info[k]["metrics"] for k in ks])
    rows.append(dict(dataset=ds, model="KG-HTT (3 seeds, mean ± sd)",
                     balanced_acc=f"{m.balanced_accuracy.mean():.3f} ± {m.balanced_accuracy.std():.3f}",
                     macro_f1=f"{m.macro_f1.mean():.3f}", log_loss=f"{m.log_loss.mean():.3f}",
                     auc=f"{m.macro_auc_ovr.mean():.4f}"))
    pe = np.mean([info[k]["p"] for k in ks], axis=0)
    rows.append(dict(dataset=ds, model="KG-HTT ensemble of 3 seeds",
                     balanced_acc=f"{balanced_accuracy_score(y, pe.argmax(1)):.3f}",
                     macro_f1=f"{f1_score(y, pe.argmax(1), average='macro'):.3f}",
                     log_loss=f"{log_loss(y, pe, labels=range(C)):.3f}",
                     auc=f"{np.nanmean(A.auc_per_class(y, pe)):.4f}"))
    for b, label in (("lr", "Logistic regression"), ("xgb", "XGBoost"), ("rf", "Random forest")):
        res = baseline(ds, b)
        if res:
            bm = res["metrics"]
            rows.append(dict(dataset=ds, model=label, balanced_acc=f"{bm['balanced_accuracy']:.3f}",
                             macro_f1=f"{bm['macro_f1']:.3f}", log_loss=f"{bm['log_loss']:.3f}",
                             auc=f"{bm['macro_auc_ovr']:.4f}"))
performance = pd.DataFrame(rows).set_index(["dataset", "model"])
performance.to_csv(f"{OUT}/1_performance.csv")
performance

### Per-class F1

Two kinds of error. Some classes change a lot from seed to seed (large `sd`): the ensemble recovers
most of them. Others stay below logistic regression in every seed: those are a real limit of the
model. The 10 hardest classes are shown; the full table is saved as CSV.

In [ ]:
per_class = {}
for ds in DATASETS:
    ks = keys_of(ds)
    classes, y = info[ks[0]]["classes"], info[ks[0]]["y"]
    t = pd.DataFrame({f"s{info[k]['seed']}": pd.Series(info[k]["metrics"]["per_class_f1"]) for k in ks})
    t["mean"], t["sd"] = t.mean(axis=1), t.std(axis=1)
    pe = np.mean([info[k]["p"] for k in ks], axis=0)
    t["ensemble"] = pd.Series(f1_score(y, pe.argmax(1), average=None), index=classes)
    lr = baseline(ds, "lr")
    if lr:
        t["logistic_reg"] = pd.Series(lr["metrics"]["per_class_f1"])
        t["gap_vs_lr"] = t["mean"] - t["logistic_reg"]
    per_class[ds] = t.sort_values("mean")
    per_class[ds].to_csv(f"{OUT}/1_per_class_f1_{ds}.csv")
    print(f"\n{ds.upper()} — 10 hardest classes")
    print(per_class[ds].head(10).round(3).to_string())

### Confusion matrices

Rows are the true class and are normalised to sum to 1, so the diagonal is the recall of each class.

In [ ]:
def plot_confusions(ds):
    ks = keys_of(ds)
    classes = info[ks[0]]["classes"]
    mats = [(f"KG-HTT seed {info[k]['seed']}", info[k]["confusion"]) for k in ks]
    lr = baseline(ds, "lr")
    if lr:
        mats.append(("Logistic regression", lr["confusion"]))
    fig, axes = plt.subplots(1, len(mats), figsize=(5.5 * len(mats), 6))
    for ax, (title, cm) in zip(np.atleast_1d(axes), mats):
        ax.imshow(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
        ax.set_title(title, fontsize=10); ax.set_xlabel("predicted"); ax.set_ylabel("true")
        ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(classes, rotation=90, fontsize=5); ax.set_yticklabels(classes, fontsize=5)
    plt.suptitle(f"{ds.upper()}", y=1.01)
    plt.tight_layout(); plt.savefig(f"{OUT}/1_confusion_{ds}.png", dpi=130, bbox_inches="tight"); plt.show()

for ds in DATASETS:
    plot_confusions(ds)

Where the errors go, for the classes that are confused most (counts, one table per seed).
The errors are always between biologically related classes: neighbouring organs, similar tissue
composition, same histology, or tumour grade.

In [ ]:
GROUPS = {"gtex": ["Colon - Sigmoid", "Colon - Transverse", "Esophagus - Gastroesophageal Junction",
                   "Esophagus - Muscularis", "Esophagus - Mucosa", "Stomach", "Breast - Mammary Tissue",
                   "Adipose - Subcutaneous", "Adipose - Visceral (Omentum)", "Skin - Not Sun Exposed (Suprapubic)",
                   "Skin - Sun Exposed (Lower Leg)", "Artery - Coronary", "Artery - Aorta", "Artery - Tibial"],
          "tcga": ["Esophageal Carcinoma", "Stomach Adenocarcinoma", "Lung Adenocarcinoma",
                   "Lung Squamous Cell Carcinoma", "Glioblastoma Multiforme", "Brain Lower Grade Glioma",
                   "Cervical & Endocervical Cancer", "Uterine Corpus Endometrioid Carcinoma"]}
for ds in DATASETS:
    classes = info[keys_of(ds)[0]]["classes"]
    sel = [classes.index(c) for c in GROUPS.get(ds, []) if c in classes]
    if not sel:
        continue
    print(f"\n===== {ds.upper()} — rows = true class, columns = predicted =====")
    for k in keys_of(ds):
        cm = info[k]["confusion"][np.ix_(sel, sel)]
        print(f"\nseed {info[k]['seed']}")
        print(pd.DataFrame(cm, index=[classes[i][:28] for i in sel],
                           columns=[classes[i][:12] for i in sel]).to_string())

## 2. Pathway permutation importance

For each pathway, its genes are shuffled across samples **only inside that pathway**: the copies of a
shared gene in the other pathways are not touched. We measure how much the macro AUC drops (and the
log-loss rises). The per-class AUC drops are kept too.

In [ ]:
imp, perclass = {}, {}
for key in RUNS:
    fcsv, fnpy, fjson = (f"{OUT}/B1_importance_{key}.csv", f"{OUT}/B1_perclass_{key}.npy",
                         f"{OUT}/B1_baseline_{key}.json")
    if not os.path.exists(fcsv):
        t0 = time.time()
        df, pc, base, _ = A.pathway_importance(get_run(key), n_perm=N_PERM, batch_size=BATCH)
        df.to_csv(fcsv); np.save(fnpy, pc)
        json.dump({k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in base.items() if k != "p0"},
                  open(fjson, "w"))
        print(f"{key}: computed in {(time.time() - t0) / 60:.0f} min")
    imp[key], perclass[key] = pd.read_csv(fcsv, index_col=0), np.load(fnpy)
K = len(next(iter(imp.values())))

size = pd.DataFrame({key: dict(largest_auc_drop=d["auc_drop"].max(), median_auc_drop=d["auc_drop"].median(),
                               pathways_above_2se=int((d["auc_drop"] > 2 * d["auc_drop_se"]).sum()),
                               largest_logloss_rise=d["logloss_rise"].max())
                     for key, d in imp.items()}).T.astype({"pathways_above_2se": int})
print(f"How large are the effects? ({K} pathways)")
print(size.to_string(formatters={"largest_auc_drop": "{:.1e}".format, "median_auc_drop": "{:.1e}".format,
                                 "largest_logloss_rise": "{:.1e}".format}))

### Do the three seeds agree?

If the importance reflected the biology, the three models of the same dataset should rank the
pathways in a similar way. Rank correlation between seeds, overlap of their top-20 pathways (two
random top-20 lists would share about 20²/K pathways), and the same check inside each class.

In [ ]:
TOP_N = 20
rows = []
for ds in DATASETS:
    for a, b in itertools.combinations(keys_of(ds), 2):
        rows.append(dict(dataset=ds, seeds=f"{info[a]['seed']} vs {info[b]['seed']}",
                         rank_corr_auc=spearmanr(imp[a]["auc_drop"], imp[b]["auc_drop"]).statistic,
                         rank_corr_logloss=spearmanr(imp[a]["logloss_rise"], imp[b]["logloss_rise"]).statistic,
                         top20_overlap=len(set(imp[a]["auc_drop"].nlargest(TOP_N).index)
                                           & set(imp[b]["auc_drop"].nlargest(TOP_N).index))))
stability = pd.DataFrame(rows)
stability.to_csv(f"{OUT}/2_seed_stability.csv", index=False)
print(stability.round(3).to_string(index=False))
print(f"expected top-20 overlap by chance: {TOP_N ** 2 / K:.1f}")

print("\nInside each class (mean rank correlation between seeds):")
for ds in DATASETS:
    ks, classes = keys_of(ds), info[keys_of(ds)[0]]["classes"]
    rho = pd.Series({c: np.nanmean([spearmanr(perclass[a][:, j], perclass[b][:, j]).statistic
                                    for a, b in itertools.combinations(ks, 2)])
                     for j, c in enumerate(classes)})
    print(f"  {ds}: median {rho.median():.2f}, max {rho.max():.2f}; "
          f"{int(rho.isna().sum())} classes never affected by any single pathway (always perfectly separated)")

**Reading.** The effects are tiny and the ranking changes from seed to seed, also inside each class.
No single pathway is needed by the model: the same information is also available in other pathways
(genes shared between pathways and correlated expression).

## 3. Gene pooling weights (beta)

Each pathway vector is a weighted average of its genes, with weights learned by attention pooling.
Normalised entropy: 1 = all genes get the same weight, i.e. the plain mean.

In [ ]:
beta, entropy = {}, {}
for key in RUNS:
    fe, fb = f"{OUT}/C1_beta_entropy_{key}.csv", f"{OUT}/C1_beta_mean_{key}.npz"
    if not (os.path.exists(fe) and os.path.exists(fb)):
        b, e = A.gene_pooling_weights(get_run(key), batch_size=BATCH)
        e.to_csv(fe); np.savez_compressed(fb, **b)
    entropy[key] = pd.read_csv(fe, index_col=0).iloc[:, 0]
    beta[key] = dict(np.load(fb, allow_pickle=True))

print(pd.DataFrame({k: dict(min_entropy=e.min(), median_entropy=e.median()) for k, e in entropy.items()}).T
      .round(4).to_string())
print("\nExample (tcga, seed 0): every gene weighs 1/n")
for p in list(beta["tcga_s0"])[:4]:
    w = beta["tcga_s0"][p]
    print(f"  {p[:45]:47s} n={len(w):3d}  1/n={1 / len(w):.4f}  weights from {w.min():.4f} to {w.max():.4f}")

**Reading.** The weights are uniform: the learned pooling does exactly what the plain mean did.

## 4. Self-attention weights

On Sacurine the attention weights were almost uniform. Here we check the same on the large datasets,
at both levels: genes inside a pathway (local) and pathways between them (global).
Normalised entropy: 1 = uniform.

In [ ]:
attn = {}
for key in RUNS:
    f = f"{OUT}/E_attention_{key}.npz"
    if not os.path.exists(f):
        a = A.attention_summary(get_run(key))
        np.savez_compressed(f, names=np.array(a["received"].index), classes=np.array(a["received_by_class"].index),
                            global_entropy=a["global_entropy"], global_max=a["global_max"],
                            received=a["received"].values, received_by_class=a["received_by_class"].values,
                            local_entropy=a["local_entropy"].values)
    d = np.load(f, allow_pickle=True)
    names = list(d["names"])
    attn[key] = dict(global_entropy=d["global_entropy"], global_max=d["global_max"],
                     received=pd.Series(d["received"], index=names),
                     local_entropy=pd.Series(d["local_entropy"], index=names))

rows = {}
for key, a in attn.items():
    rv = a["received"]
    rows[key] = dict(global_entropy=a["global_entropy"].mean(), local_entropy_median=a["local_entropy"].median(),
                     largest_weight=a["global_max"].mean(), uniform_weight=1 / len(rv),
                     received_cv=rv.std() / rv.mean(),
                     received_vs_size=spearmanr(rv, imp[key]["n_genes"].reindex(rv.index)).statistic)
summary_attn = pd.DataFrame(rows).T
summary_attn.to_csv(f"{OUT}/4_attention_summary.csv")
print("received_cv: spread of the attention received by the pathways (Sacurine: about 0.10)")
print("received_vs_size: rank correlation between attention received and pathway size")
print(summary_attn.round(4).to_string())
for ds in DATASETS:
    r = [spearmanr(attn[a]["received"], attn[b]["received"]).statistic
         for a, b in itertools.combinations(keys_of(ds), 2)]
    print(f"{ds}: attention received, rank correlation between seeds {min(r):.2f}–{max(r):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ds in DATASETS:
    ks = keys_of(ds)
    axes[0].hist(np.concatenate([attn[k]["global_entropy"] for k in ks]), bins=40, alpha=.6, label=ds)
    axes[1].hist(pd.concat([attn[k]["local_entropy"] for k in ks]), bins=40, alpha=.6, label=ds)
axes[0].set_xlabel("global attention: normalised entropy (per sample)")
axes[1].set_xlabel("local attention: normalised entropy (per pathway)")
for ax in axes:
    ax.set_xlim(0, 1.02); ax.axvline(1, color="k", ls="--", lw=.8); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/4_attention_entropy.png", dpi=130); plt.show()

**Reading.** Both levels are almost uniform, as on Sacurine: every gene attends to all the genes of
its pathway, and every pathway to all the pathways, with nearly equal weights.

## 5. Conclusions

- **Accuracy.** KG-HTT reaches 0.944 (TCGA) and 0.926 (GTEx) balanced accuracy: on par with random
  forest, below logistic regression and XGBoost. The ensemble of three seeds recovers about half of
  the gap.
- **Errors.** They are between biologically related classes (neighbouring organs, similar tissue,
  same histology, tumour grade).
- **Pathway importance.** Very small effects, and different pathways for different seeds.
- **Attention.** Gene pooling weights and self-attention weights are uniform at both levels, as on
  Sacurine.
- **Overall.** The model spreads the information evenly across genes and pathways; with this
  architecture, attention and single-pathway permutation do not give a stable attribution.